# How to turn on mixed-precision training - line by line 🔌
### A small, hands-on notebook: start from a normal training loop and switch mixed precision on, one line at a time

You already know *why* (notebook `02_mixed_precision_training`). This notebook is only about **how**: *which lines do you add to your training loop, and what does each one do?*

**The short answer.** You add **three things** to a normal loop:

| # | add | when | what it gives you |
|---|---|---|---|
| ① | `with torch.autocast(...):` around the forward pass | always | matmuls run in bf16/fp16 |
| ② | `GradScaler` (`scale → step → update`) | **fp16 only** | gradients do not underflow |
| ③ | `scaler.unscale_(opt)` before gradient clipping | only if you clip, **fp16 only** | clipping sees the true gradient size |

Nothing else changes: the model, the optimizer, the data and the learning rate stay the same.

| step | content |
|---|---|
| 1 | the normal fp32 loop, every line explained |
| 2 | ① bf16: add `autocast` - and **prove** it is on |
| 3 | ② fp16: add `GradScaler` - watch the scale, a skipped step |
| 4 | ③ add gradient clipping correctly |
| 5 | the one universal loop (the template to copy) |
| 6 | the same thing as a config switch in this repo (`AMPManager`) |
| 7 | one-liners in Hugging Face Trainer / Lightning / Accelerate |
| 8 | "is it really on?" and common errors |

Every cell below is executed and checked (`PASS` lines); CPU only.

In [1]:
import math, sys, importlib.util, types
from pathlib import Path
import numpy as np, torch, torch.nn as nn

torch.manual_seed(0)
CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append((name, bool(ok), detail))
    print(("PASS" if ok else "FAIL"), "-", name, ("| " + detail) if detail else "")

def toy(seed=0):
    """A tiny regression problem: 16 inputs -> 1 output. Same for every loop below."""
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(16, 64), nn.GELU(), nn.Linear(64, 64), nn.GELU(), nn.Linear(64, 1))
    x = torch.randn(512, 16)
    y = (x[:, :1] * 2 - x[:, 1:2]) + 0.1 * torch.randn(512, 1)
    return model, x, y
print("torch", torch.__version__)

torch 2.14.1


## Step 1 - The normal (fp32) loop, every line explained

This is the loop we will *modify*. Read it once carefully - the other loops differ from it by only a few lines.

In [2]:
def train_fp32(steps=100):
    model, x, y = toy()                                          # (1)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)         # (2)
    losses = []                                                  # (3)
    for step in range(steps):                                    # (4)
        opt.zero_grad(set_to_none=True)                          # (5)
        pred = model(x)                                          # (6)
        loss = nn.functional.mse_loss(pred, y)                   # (7)
        loss.backward()                                          # (8)
        opt.step()                                               # (9)
        losses.append(loss.item())                               # (10)
    return np.array(losses)

l32 = train_fp32()
print(f"loss {l32[0]:.3f} -> {l32[-1]:.4f}")
check("baseline fp32 loop trains", l32[-1] < l32[0] / 5, f"{l32[0]:.3f} -> {l32[-1]:.4f}")

loss 4.884 -> 0.0151
PASS - baseline fp32 loop trains | 4.884 -> 0.0151


| line | what it does | why it matters here |
|---|---|---|
| **(1)** `model, x, y = toy()` | builds the network and the data. All weights are created as **fp32** (4 bytes each) - PyTorch's default dtype. | These fp32 weights will become the *master weights* in mixed precision. We never convert them. |
| **(2)** `opt = AdamW(model.parameters(), lr=3e-3)` | creates the optimizer. It keeps its own fp32 state (two extra numbers per weight). | The optimizer updates the **fp32** weights, which is exactly what we want to keep. |
| **(3)** `losses = []` | a list to record the loss per step. | bookkeeping only. |
| **(4)** `for step in range(steps)` | one iteration = one optimizer step on the whole (tiny) data set. | - |
| **(5)** `opt.zero_grad(set_to_none=True)` | clears the gradients left over from the previous step. `set_to_none=True` frees the memory instead of filling it with zeros (faster). | Gradients *accumulate* in PyTorch, so forgetting this adds old gradients to new ones. |
| **(6)** `pred = model(x)` | the **forward pass**: pushes `x` through the layers. This is where nearly all the arithmetic (matrix multiplies) happens. | **This is the line mixed precision speeds up.** |
| **(7)** `loss = mse_loss(pred, y)` | one number saying how wrong `pred` is: the mean of `(pred − y)²`. | It is an average over many values, so we will keep it in fp32. |
| **(8)** `loss.backward()` | the **backward pass**: computes `d loss / d weight` for every weight and stores it in `weight.grad`. | The second big compute cost. Runs in whatever dtype the forward used. |
| **(9)** `opt.step()` | uses `weight.grad` to update each weight: `w ← w − lr · (adjusted gradient)`. | Must happen in fp32 (tiny updates, see notebook 02 Step 3). |
| **(10)** `losses.append(loss.item())` | `.item()` turns the 1-element tensor into a Python float. | - |

Everything we add next goes around lines **(6)-(9)**.

## Step 2 - ① bf16: add `torch.autocast`  (the simplest way to turn it on)

Here is the *entire* change, as a diff. Only **two lines** are different:

```diff
   for step in range(steps):
       opt.zero_grad(set_to_none=True)
-      pred = model(x)
-      loss = nn.functional.mse_loss(pred, y)
+      with torch.autocast(device_type="cpu", dtype=torch.bfloat16):      # ← ① NEW
+          pred = model(x)                                                 #   (indented)
+      loss = nn.functional.mse_loss(pred.float(), y)                      # ← .float() NEW
       loss.backward()
       opt.step()
```

In [3]:
def train_bf16(steps=100):
    model, x, y = toy()
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    losses = []
    for step in range(steps):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cpu", dtype=torch.bfloat16):    # (A) switch mixed precision ON for everything indented below
            pred = model(x)                                              # (B) forward pass, now using bf16 matmuls
        loss = nn.functional.mse_loss(pred.float(), y)                   # (C) back to fp32 for the loss
        loss.backward()                                                  # (D) unchanged
        opt.step()                                                       # (E) unchanged: still updates fp32 weights
        losses.append(loss.item())
    return np.array(losses)

lb = train_bf16()
print(f"loss {lb[0]:.3f} -> {lb[-1]:.4f}   (fp32 run ended at {l32[-1]:.4f})")
check("bf16 autocast loop trains", lb[-1] < lb[0] / 5, f"{lb[0]:.3f} -> {lb[-1]:.4f}")

loss 4.884 -> 0.0151   (fp32 run ended at 0.0151)
PASS - bf16 autocast loop trains | 4.884 -> 0.0151


### What each new line means

**(A) `with torch.autocast(device_type="cpu", dtype=torch.bfloat16):`**
- `torch.autocast(...)` creates a *context*: a region of code with a changed rule.
- `with ...:` means "apply the rule to every line indented below, and restore the normal rule when the block ends". Outside the block nothing is different.
- `device_type="cpu"` - which hardware's operations to change: `"cpu"`, `"cuda"` (NVIDIA **and** AMD ROCm both use `"cuda"`), or `"mps"` (Apple). It must match where your tensors live.
- `dtype=torch.bfloat16` - the *cheap* dtype to use. Use `torch.float16` for fp16 (then you also need the scaler, Step 3).
- The rule: *for each operation, autocast looks up a list. Heavy operations (matmul, linear, convolution) run in the cheap dtype; precision-sensitive ones (sums, losses, softmax…) stay fp32.* Inputs are cast on the fly - **your model's weights are not changed**.

**(B) `pred = model(x)`** - the identical line as before, now *inside* the block. Each `nn.Linear` casts its fp32 weight and input to bf16 for the multiply (the "cheap copy" of notebook 02) and returns a bf16 result.

**(C) `mse_loss(pred.float(), y)`** - `pred` is bf16 now. `.float()` converts it to fp32 so that the squaring and averaging - and therefore the number that `backward()` differentiates - are done precisely. (Many built-in losses do this automatically under autocast; writing `.float()` makes it explicit and safe.)

**(D) `loss.backward()`** - **unchanged, and outside the block.** You do not wrap `backward` in autocast: PyTorch automatically runs each backward operation in the same dtype as its forward operation.

**(E) `opt.step()`** - **unchanged.** The weights it updates are still the fp32 originals, so this *is* the master-weights update.

### Prove it is on
Don't trust that autocast "worked" - look at the dtypes. We attach a small observer to every `Linear` layer and also look at the weights, the loss and the gradients:

In [4]:
model, x, y = toy()
seen = []
hooks = [m.register_forward_hook(lambda mod, inp, out: seen.append((f"Linear {tuple(mod.weight.shape)}", inp[0].dtype, out.dtype)))
         for m in model if isinstance(m, nn.Linear)]

print("autocast enabled outside the block? ", torch.is_autocast_enabled("cpu"))
with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    print("autocast enabled inside the block?  ", torch.is_autocast_enabled("cpu"), "| cheap dtype:", torch.get_autocast_dtype("cpu"))
    pred = model(x)
loss = nn.functional.mse_loss(pred.float(), y); loss.backward()
for h in hooks: h.remove()

print(f"\n{'layer':22s}{'input dtype':>15s}{'output dtype':>15s}")
for n_, di, do in seen: print(f"{n_:22s}{str(di):>15s}{str(do):>15s}")
print("\nweights dtype   :", model[0].weight.dtype, "   <- the master weights are untouched")
print("pred dtype      :", pred.dtype)
print("loss dtype      :", loss.dtype)
print("gradient dtype  :", model[0].weight.grad.dtype, "   <- gradients land on the fp32 weights, so they are fp32 too")

# numerically different from fp32, but only slightly -> it really did compute in lower precision
with torch.no_grad(): p32 = model(x)
rel = ((pred.float() - p32).norm() / p32.norm()).item()
print(f"\nrelative difference bf16-autocast vs fp32 prediction: {rel:.4f}")
check("outside the block autocast is off, inside it is on (bf16)", (not torch.is_autocast_enabled("cpu")) and torch.get_autocast_dtype("cpu") == torch.bfloat16)
check("the first Linear gets fp32 data in; every Linear outputs bf16", seen[0][1] == torch.float32 and all(do == torch.bfloat16 for _, _, do in seen))
check("later Linears already receive bf16 activations (the cast happens once, then stays cheap)", all(di == torch.bfloat16 for _, di, _ in seen[1:]))
check("weights, loss and gradients are fp32", model[0].weight.dtype == loss.dtype == model[0].weight.grad.dtype == torch.float32)
check("bf16 prediction differs from fp32 by ~0.1-3% (it really used bf16)", 1e-3 < rel < 3e-2, f"{rel:.4f}")

autocast enabled outside the block?  False
autocast enabled inside the block?   True | cheap dtype: torch.bfloat16

layer                     input dtype   output dtype
Linear (64, 16)         torch.float32 torch.bfloat16
Linear (64, 64)        torch.bfloat16 torch.bfloat16
Linear (1, 64)         torch.bfloat16 torch.bfloat16

weights dtype   : torch.float32    <- the master weights are untouched
pred dtype      : torch.bfloat16
loss dtype      : torch.float32
gradient dtype  : torch.float32    <- gradients land on the fp32 weights, so they are fp32 too

relative difference bf16-autocast vs fp32 prediction: 0.0029
PASS - outside the block autocast is off, inside it is on (bf16) 
PASS - the first Linear gets fp32 data in; every Linear outputs bf16 
PASS - later Linears already receive bf16 activations (the cast happens once, then stays cheap) 
PASS - weights, loss and gradients are fp32 
PASS - bf16 prediction differs from fp32 by ~0.1-3% (it really used bf16) | 0.0029


That table is the whole trick in one picture: **your fp32 data enters the first layer, every layer then outputs bf16 (so later layers receive bf16), while weights, gradients and the loss stay fp32.** For bf16 you are done - no scaler needed, because bf16 has the same range as fp32.

## Step 3 - ② fp16: add `GradScaler`

fp16 has a much smaller range, so small gradients can underflow to 0. `GradScaler` fixes that with *loss scaling* (notebook 02, Steps 4-5). The change to the loop - the autocast lines are the same as Step 2 except `dtype=torch.float16`:

```diff
+ scaler = torch.amp.GradScaler("cpu")                  # ② create ONCE, before the loop
   for step in range(steps):
       opt.zero_grad(set_to_none=True)
       with torch.autocast(device_type="cpu", dtype=torch.float16):
           pred = model(x)
       loss = nn.functional.mse_loss(pred.float(), y)
-      loss.backward()
-      opt.step()
+      scaler.scale(loss).backward()                     # ② backward on (loss × S)
+      scaler.step(opt)                                  # ② unscale, check for inf/NaN, then opt.step() (or skip)
+      scaler.update()                                   # ② adapt S for the next step
```

In [5]:
def train_fp16(steps=100):
    model, x, y = toy()
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    scaler = torch.amp.GradScaler("cpu")                                 # (A) create once: holds the loss scale S (starts at 65536)
    losses = []
    for step in range(steps):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cpu", dtype=torch.float16):
            pred = model(x)
        loss = nn.functional.mse_loss(pred.float(), y)
        scaler.scale(loss).backward()                                    # (B) multiply the loss by S, then backward
        scaler.step(opt)                                                 # (C) divide grads by S; if all finite -> opt.step(), else skip
        scaler.update()                                                  # (D) shrink S after a skip, grow it after many good steps
        losses.append(loss.item())
    return np.array(losses), scaler.get_scale()

lh, S_end = train_fp16()
print(f"loss {lh[0]:.3f} -> {lh[-1]:.4f}   final scale S = {S_end:g}")
check("fp16 autocast + GradScaler loop trains", lh[-1] < lh[0] / 5, f"{lh[0]:.3f} -> {lh[-1]:.4f}")

loss 4.884 -> 0.0151   final scale S = 65536
PASS - fp16 autocast + GradScaler loop trains | 4.884 -> 0.0151


### What each scaler line means

**(A) `scaler = torch.amp.GradScaler("cpu")`** - creates the scaler **once, outside the loop** (it has to remember `S` between steps). The argument is the device type. `S` starts at `2¹⁶ = 65536`. Optional arguments: `init_scale`, `growth_factor` (default 2), `backoff_factor` (0.5), `growth_interval` (2000 steps). Pass `enabled=False` and it turns into a harmless pass-through (handy, Step 5).

**(B) `scaler.scale(loss).backward()`** - `scaler.scale(loss)` returns `loss × S`. Calling `.backward()` on it makes *every* gradient `S` times bigger, which lifts tiny gradients out of fp16's underflow zone.

**(C) `scaler.step(opt)`** - replaces `opt.step()`. Inside it: (1) divides all gradients by `S` again (so the optimizer sees true gradients), (2) checks them for `inf`/`NaN`, (3) if all finite calls `opt.step()`, otherwise **skips the update** so one bad batch cannot corrupt the weights.

**(D) `scaler.update()`** - once per step, after `step`. If the step was skipped it halves `S`; after `growth_interval` good steps in a row it doubles `S`.

Let's watch these happen with real numbers instead of taking the words on trust:

In [6]:
model, x, y = toy(); opt = torch.optim.SGD(model.parameters(), lr=0.01)
scaler = torch.amp.GradScaler("cpu", init_scale=1024.0)

# --- (B) scale() multiplies the loss by S
with torch.autocast("cpu", dtype=torch.float16): pred = model(x)
loss = nn.functional.mse_loss(pred.float(), y)
scaled = scaler.scale(loss)
print(f"loss = {loss.item():.4f}   scaler.scale(loss) = {scaled.item():.2f}   ratio = {scaled.item() / loss.item():.0f}  (S = {scaler.get_scale():.0f})")
check("scaler.scale(loss) == loss * S", math.isclose(scaled.item() / loss.item(), 1024.0, rel_tol=1e-5))

# --- the gradients are S times larger after the scaled backward, and step() divides them back
scaled.backward(); g_scaled = model[0].weight.grad.clone()
scaler.unscale_(opt); g_true = model[0].weight.grad.clone()           # what step() does internally, done by hand to look at it
print(f"mean |grad| scaled = {g_scaled.abs().mean():.3e}   after unscale = {g_true.abs().mean():.3e}   ratio = {g_scaled.abs().mean() / g_true.abs().mean():.0f}")
check("gradients are S times bigger before unscale", math.isclose((g_scaled.abs().mean() / g_true.abs().mean()).item(), 1024.0, rel_tol=1e-4))

# --- (C)+(D) an overflow: poison one gradient with inf and see what step()/update() do
scaler2 = torch.amp.GradScaler("cpu", init_scale=1024.0); w_before = model[0].weight.detach().clone()
opt.zero_grad()
with torch.autocast("cpu", dtype=torch.float16): pred = model(x)
scaler2.scale(nn.functional.mse_loss(pred.float(), y)).backward()
model[0].weight.grad[0, 0] = float("inf")                              # simulate an fp16 overflow in one gradient
scaler2.step(opt); scaler2.update()
changed = not torch.equal(model[0].weight.detach(), w_before)
print(f"after an inf gradient:  weights changed = {changed},   scale 1024 -> {scaler2.get_scale():.0f}")
check("an inf gradient makes step() SKIP the update (weights unchanged)", not changed)
check("...and update() halves the scale (1024 -> 512)", scaler2.get_scale() == 512.0)

loss = 4.8842   scaler.scale(loss) = 5001.38   ratio = 1024  (S = 1024)
PASS - scaler.scale(loss) == loss * S 
mean |grad| scaled = 4.869e+00   after unscale = 4.755e-03   ratio = 1024
PASS - gradients are S times bigger before unscale 
after an inf gradient:  weights changed = False,   scale 1024 -> 512
PASS - an inf gradient makes step() SKIP the update (weights unchanged) 
PASS - ...and update() halves the scale (1024 -> 512) 


Both behaviours are the ones promised: scaled by `S`, unscaled before the update, and **skip + halve `S` on overflow**. In a long run you will see `S` start at 65536, drop a few times in the first steps, then sit around a stable value.

**Which one to use?** bf16 if your hardware supports it (no scaler, fewer surprises); fp16 + scaler otherwise. On NVIDIA/AMD GPUs `torch.cuda.is_bf16_supported()` tells you; Step 5 uses it.

## Step 4 - ③ Gradient clipping: call `unscale_` first (fp16 only)

Clipping caps the *size* of the gradient vector (`clip_grad_norm_(params, max_norm)`), which stabilises training. With a scaler the gradients are still multiplied by `S` at that point, so you would clip the wrong thing - `S` times too large. The fix is one line:

```diff
   scaler.scale(loss).backward()
+  scaler.unscale_(opt)                                   # ③ gradients are now the true ones
+  torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
   scaler.step(opt)
   scaler.update()
```
Measured:

In [7]:
model, x, y = toy(); opt = torch.optim.SGD(model.parameters(), lr=0.01); S = 1024.0
scaler = torch.amp.GradScaler("cpu", init_scale=S)
with torch.autocast("cpu", dtype=torch.float16): pred = model(x)
scaler.scale(nn.functional.mse_loss(pred.float(), y)).backward()

norm_without = torch.nn.utils.clip_grad_norm_(model.parameters(), 1e9).item()   # huge max_norm = just measure the norm
scaler.unscale_(opt)                                                              # ③
norm_with = torch.nn.utils.clip_grad_norm_(model.parameters(), 1e9).item()
print(f"gradient norm seen by clip_grad_norm_ without unscale_: {norm_without:.2f}")
print(f"gradient norm seen by clip_grad_norm_ after  unscale_: {norm_with:.4f}   (ratio {norm_without / norm_with:.0f} = S)")
check("without unscale_, clipping sees a norm S times too large", math.isclose(norm_without / norm_with, S, rel_tol=1e-4))

# Unscaling twice would be a bug; PyTorch protects you:
try: scaler.unscale_(opt); ok = False
except RuntimeError as e: ok = "unscale_" in str(e); print("calling unscale_ twice ->", str(e)[:70])
check("calling unscale_ twice in one step raises an error (one unscale_ per step)", ok)

gradient norm seen by clip_grad_norm_ without unscale_: 1089.72
gradient norm seen by clip_grad_norm_ after  unscale_: 1.0642   (ratio 1024 = S)
PASS - without unscale_, clipping sees a norm S times too large 
calling unscale_ twice -> unscale_() has already been called on this optimizer since the last up
PASS - calling unscale_ twice in one step raises an error (one unscale_ per step) 


Rules of thumb: **one `unscale_` per optimizer step**, **before** clipping, **before** `scaler.step`. With bf16 there is no scaler, so you simply clip as you always did.

## Step 5 - The universal loop: one function for fp32, bf16 and fp16

In real code you want to switch precision with a flag, not edit the loop. The trick: a **disabled** `GradScaler` and a **disabled** `autocast` are no-ops, so a single loop covers all three modes. *This is the template to copy.*

In [8]:
def pick_precision(device_type="cpu"):
    """Choose (autocast dtype, use_scaler) for a device. None dtype = plain fp32."""
    if device_type == "cuda":                                  # NVIDIA and AMD-ROCm both report "cuda"
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported(): return torch.bfloat16, False
        return torch.float16, True
    if device_type in ("cpu", "mps"): return torch.bfloat16, False
    return None, False

def train(device_type="cpu", amp_dtype=torch.bfloat16, steps=100, clip=1.0, lr=3e-3):
    model, x, y = toy()                                                          # (1) model stays fp32
    opt = torch.optim.AdamW(model.parameters(), lr=lr)                           # (2)
    use_amp = amp_dtype is not None                                              # (3) None -> plain fp32
    use_scaler = amp_dtype == torch.float16                                      # (4) only fp16 needs loss scaling
    scaler = torch.amp.GradScaler(device_type, enabled=use_scaler)               # (5) disabled scaler = pass-through
    losses = []
    for step in range(steps):
        opt.zero_grad(set_to_none=True)                                          # (6)
        with torch.autocast(device_type=device_type, dtype=amp_dtype or torch.float32, enabled=use_amp):   # (7)
            pred = model(x)                                                      # (8)
        loss = nn.functional.mse_loss(pred.float(), y)                           # (9)
        scaler.scale(loss).backward()                                            # (10) == loss.backward() when the scaler is disabled
        scaler.unscale_(opt)                                                     # (11) no-op when disabled
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)                 # (12)
        scaler.step(opt)                                                         # (13) == opt.step() when disabled
        scaler.update()                                                          # (14) no-op when disabled
        losses.append(loss.item())                                               # (15)
    return np.array(losses)

print("pick_precision('cpu') ->", pick_precision("cpu"))
runs = {"fp32": train(amp_dtype=None), "bf16": train(amp_dtype=torch.bfloat16), "fp16": train(amp_dtype=torch.float16)}
for k, l in runs.items(): print(f"{k:5s} loss {l[0]:.3f} -> {l[-1]:.5f}")
for k, l in runs.items(): check(f"universal loop, {k}: trains", l[-1] < l[0] / 5, f"{l[0]:.3f} -> {l[-1]:.4f}")
d_bf, d_h = np.abs(runs["fp32"] - runs["bf16"]).max(), np.abs(runs["fp32"] - runs["fp16"]).max()
print(f"largest step-by-step loss difference vs fp32:  bf16 {d_bf:.1e}   fp16 {d_h:.1e}   (nonzero: the curves are genuinely different computations)")
check("the three loss CURVES differ (autocast really changes the arithmetic) yet end together", d_bf > 1e-5 and d_h > 1e-5, f"{d_bf:.1e} / {d_h:.1e}")
check("all three end within 15% of the fp32 loss", all(abs(l[-1] - runs['fp32'][-1]) / runs['fp32'][-1] < 0.15 for l in runs.values()), " / ".join(f"{l[-1]:.4f}" for l in runs.values()))

pick_precision('cpu') -> (torch.bfloat16, False)


fp32  loss 4.884 -> 0.01099
bf16  loss 4.884 -> 0.01098
fp16  loss 4.884 -> 0.01099
PASS - universal loop, fp32: trains | 4.884 -> 0.0110
PASS - universal loop, bf16: trains | 4.884 -> 0.0110
PASS - universal loop, fp16: trains | 4.884 -> 0.0110
largest step-by-step loss difference vs fp32:  bf16 1.1e-03   fp16 2.7e-04   (nonzero: the curves are genuinely different computations)
PASS - the three loss CURVES differ (autocast really changes the arithmetic) yet end together | 1.1e-03 / 2.7e-04
PASS - all three end within 15% of the fp32 loss | 0.0110 / 0.0110 / 0.0110


**Line by line (only what is new compared with the loops above):**

| line | explanation |
|---|---|
| **(3)** `use_amp = amp_dtype is not None` | `None` means "no mixed precision"; the same loop then behaves as the plain fp32 loop. |
| **(4)** `use_scaler = amp_dtype == torch.float16` | the scaler is needed **only** for fp16. bf16 and fp32 skip it. |
| **(5)** `GradScaler(device_type, enabled=use_scaler)` | when `enabled=False`, `scale(loss)` returns `loss` unchanged and `step`/`update`/`unscale_` do nothing but call the plain optimizer. One code path for every mode. |
| **(7)** `torch.autocast(..., dtype=amp_dtype or torch.float32, enabled=use_amp)` | `enabled=False` makes the `with` block do nothing. The `or torch.float32` only avoids passing `None` as a dtype. |
| **(10)** `scaler.scale(loss).backward()` | with a disabled scaler this is exactly `loss.backward()`. |
| **(11)-(12)** `unscale_` then `clip_grad_norm_` | the correct order from Step 4. |
| **(13)-(14)** `scaler.step(opt)`, `scaler.update()` | with a disabled scaler `step` just calls `opt.step()`. |

Lines (1), (2), (6), (8), (9), (15) are the same as in the fp32 loop of Step 1. **That is the point: mixed precision is a thin wrapper, not a rewrite.**

*Why do all three print almost the same final loss?* The task is easy, so every run converges to the data's noise floor (~0.011); the checks above show the curves differ along the way. On a large model the differences are what you tune and monitor.

**`pick_precision`** encodes the earlier advice: bf16 wherever it exists (CPU, Apple, recent GPUs), fp16 + scaler on older GPUs. Only the CPU branch is executed here.

## Step 6 - The same thing as a *config switch* in this repo

You usually do not hand-write that loop. `HaleBlocks` (the shared training library under every Hale* repo) already contains it:

- **config**: `HaleBlocks/src/hale_core/config/sections/amp.py` → `AMPConfig` (the knobs)
- **engine**: `HaleBlocks/src/hale_core/training/amp.py` → `AMPManager` (autocast + scaler behind five methods)
- **trainer**: `HaleBlocks/src/hale_core/training/trainer/trainer.py` calls it inside its loop

To turn mixed precision on you set **config values, not code**:

```yaml
train:
  amp:
    enabled: true        # master switch (default: false = plain fp32)
    dtype: bf16          # "bf16" | "fp16" | "fp32"
    # fp16 only:
    init_scale: 65536.0
    growth_factor: 2.0
    backoff_factor: 0.5
    growth_interval: 2000
    enabled_on_cpu: false   # CPU autocast is opt-in
```

Here is what the trainer does with it - the **same pattern as our universal loop**, with each of our lines mapped to a method:

| our loop (Step 5) | `AMPManager` method used by the trainer |
|---|---|
| `with torch.autocast(...)` | `with amp.forward_context():` |
| `scaler.scale(loss).backward()` | `amp.scale_loss(loss).backward()` |
| `scaler.unscale_(opt)` | `amp.unscale(opt)` |
| `scaler.step(opt)` | `amp.step(opt)` |
| `scaler.update()` | `amp.update()` |
| `opt.zero_grad(set_to_none=True)` | `amp.zero_grad(opt)` |
| (log the scale) | `amp.stats()` → `{"amp/scale": ..., "amp/dtype": ...}` |
| (save with checkpoint) | `amp.state_dict()` / `amp.load_state_dict()` |

Let's drive the real `AMPManager` source file with the same toy problem. (We load the file directly so the notebook has no dependency on the package being installed; `AMPConfig` only matters for its field names; we use a plain namespace with the same fields and defaults.)

In [9]:
AMP_FILE = Path("../../../HaleBlocks/src/hale_core/training/amp.py").resolve()
print("AMPManager source:", AMP_FILE.relative_to(AMP_FILE.parents[5]) if AMP_FILE.exists() else "NOT FOUND")
spec = importlib.util.spec_from_file_location("hale_amp_standalone", AMP_FILE); amp_mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(amp_mod)
AMPManager = amp_mod.AMPManager

def make_cfg(**kw):
    """Same field names/defaults as hale_core's AMPConfig (a plain namespace so no pydantic is needed here)."""
    d = dict(enabled=False, dtype="bf16", init_scale=65536.0, growth_factor=2.0, backoff_factor=0.5, growth_interval=2000, enabled_on_cpu=False)
    d.update(kw); return types.SimpleNamespace(**d)

def train_with_manager(cfg, device="cpu", steps=100, clip=1.0):
    model, x, y = toy(); opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    amp = AMPManager(cfg, device)                                   # builds autocast settings (+ GradScaler for fp16)
    losses = []
    for step in range(steps):
        amp.zero_grad(opt)                                          # opt.zero_grad(set_to_none=True)
        with amp.forward_context():                                 # autocast, or nothing when disabled
            pred = model(x)
        loss = nn.functional.mse_loss(pred.float(), y)
        amp.scale_loss(loss).backward()                             # scaled only for fp16
        amp.unscale(opt)                                            # so clipping sees true gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        amp.step(opt); amp.update()
        losses.append(loss.item())
    return np.array(losses), amp

cases = {
    "default config (enabled=False)":        make_cfg(),
    "bf16 on, but enabled_on_cpu=False":     make_cfg(enabled=True, dtype="bf16"),
    "bf16 on, enabled_on_cpu=True":          make_cfg(enabled=True, dtype="bf16", enabled_on_cpu=True),
    "fp16 on, enabled_on_cpu=True":          make_cfg(enabled=True, dtype="fp16", enabled_on_cpu=True),   # the repo gate refuses this on CPU
}
mgr = {}
for name, cfg in cases.items():
    l, amp = train_with_manager(cfg); mgr[name] = (l, amp)
    print(f"{name:38s} amp.enabled={str(amp.enabled):5s} stats={amp.stats()}   loss {l[0]:.3f} -> {l[-1]:.4f}")

AMPManager source: Music/HaleBlocks/src/hale_core/training/amp.py
default config (enabled=False)         amp.enabled=False stats={'amp/enabled': 0.0, 'amp/dtype': 'bf16', 'amp/scale': 1.0}   loss 4.884 -> 0.0110
bf16 on, but enabled_on_cpu=False      amp.enabled=False stats={'amp/enabled': 0.0, 'amp/dtype': 'bf16', 'amp/scale': 1.0}   loss 4.884 -> 0.0110
bf16 on, enabled_on_cpu=True           amp.enabled=True  stats={'amp/enabled': 1.0, 'amp/dtype': 'bf16', 'amp/scale': 1.0}   loss 4.884 -> 0.0110


fp16 on, enabled_on_cpu=True           amp.enabled=False stats={'amp/enabled': 0.0, 'amp/dtype': 'fp16', 'amp/scale': 1.0}   loss 4.884 -> 0.0110


In [10]:
names = list(cases)
check("default config: AMP is OFF (plain fp32) and the loop still trains", (not mgr[names[0]][1].enabled) and mgr[names[0]][0][-1] < mgr[names[0]][0][0] / 5)
check("bf16 on CPU stays OFF unless enabled_on_cpu=True (a safety gate)", not mgr[names[1]][1].enabled)
check("bf16 + enabled_on_cpu=True: AMP active and trains", mgr[names[2]][1].enabled and mgr[names[2]][0][-1] < mgr[names[2]][0][0] / 5)
check("repo gate: fp16 is refused on CPU (AMP stays off even with enabled_on_cpu=True)", not mgr[names[3]][1].enabled and mgr[names[3]][1]._scaler is None)
check("bf16 never creates a GradScaler", mgr[names[2]][1]._scaler is None)
# state_dict round trip (what the trainer saves into a checkpoint). This machine has no CUDA, so an fp16 manager built for "cuda"
# gets a disabled scaler; we swap in a real CPU GradScaler just to exercise state_dict()/load_state_dict().
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore"); m1 = AMPManager(make_cfg(enabled=True, dtype="fp16"), "cuda"); m2 = AMPManager(make_cfg(enabled=True, dtype="fp16"), "cuda")
m1._scaler = torch.amp.GradScaler("cpu", init_scale=1024.0); m2._scaler = torch.amp.GradScaler("cpu", init_scale=123.0)
m2.load_state_dict(m1.state_dict())
check("AMPManager.state_dict()/load_state_dict() restores the loss scale (123 -> 1024)", m2._scaler.get_scale() == 1024.0)

PASS - default config: AMP is OFF (plain fp32) and the loop still trains 
PASS - bf16 on CPU stays OFF unless enabled_on_cpu=True (a safety gate) 
PASS - bf16 + enabled_on_cpu=True: AMP active and trains 
PASS - repo gate: fp16 is refused on CPU (AMP stays off even with enabled_on_cpu=True) 
PASS - bf16 never creates a GradScaler 
PASS - AMPManager.state_dict()/load_state_dict() restores the loss scale (123 -> 1024) 


Three details worth noticing in the output:

1. **Default = off.** `enabled: false` by default, so existing runs are unchanged until you opt in.
2. **CPU needs two switches** (`enabled: true` *and* `enabled_on_cpu: true`). The repo's gate deliberately limits CPU autocast to bf16; fp16 on CPU is refused outright (last case above: `amp.enabled=False` even with `enabled_on_cpu=True`). So the manager's fp16 + `GradScaler` path needs a GPU and was **not exercised end to end here**; the scaler mechanics themselves were verified with plain `GradScaler("cpu")` in Step 3.
3. **The scale is part of the checkpoint.** If you resume an fp16 run without restoring it, the scale resets to 65 536 and the first steps get skipped. The trainer handles this (`amp_state_dict` in the checkpoint).

*Not run here:* the full `Trainer` end to end - that needs a real config/dataset; the loop above is the `Trainer`'s own sequence of `AMPManager` calls (`trainer.py`, "Forward / Backward / Grad clipping / Optimizer step" blocks).

## Step 7 - Higher-level libraries: it is one flag

If you use a framework, the three additions are hidden behind one option. **These are shown for reference from their documentation and were *not* executed here (the packages are not installed in this environment).**

```python
# Hugging Face Transformers  -  the Trainer builds autocast (+ GradScaler for fp16) for you
from transformers import TrainingArguments
args = TrainingArguments(output_dir="out", bf16=True)      # or fp16=True

# PyTorch Lightning
import lightning as L
trainer = L.Trainer(precision="bf16-mixed")                # or "16-mixed" for fp16 + scaler

# Hugging Face Accelerate
from accelerate import Accelerator
accelerator = Accelerator(mixed_precision="bf16")          # or "fp16"
#   then:  accelerator.backward(loss)   instead of  loss.backward()
```
All of them do exactly what Steps 2-4 did by hand.

## Step 8 - "Is it really on?" and common errors

**A 10-second test** - run one batch and print dtypes (as in Step 2): the layer outputs should be bf16/fp16, while weights, loss and gradients stay fp32.

| symptom | usual cause | fix |
|---|---|---|
| no speed/memory change at all | `autocast` not entered, wrong `device_type`, or the device has no fast low-precision path | print `torch.is_autocast_enabled(device_type)` inside the block |
| `RuntimeError: expected scalar type Half but found Float` | you called `.half()` on the model/inputs yourself *and* use autocast, or mixed dtypes by hand | do not `.half()` the model; leave weights fp32 and let autocast cast per-op |
| loss becomes `NaN`/`inf` right away (fp16) | no scaler, or an overflow inside the model | add `GradScaler`; try bf16; lower the learning rate |
| fp16 training stalls, scale keeps halving | genuine overflow in some layer, not a scaler problem | check inputs/learning rate; use bf16 |
| clipping seems to do nothing / too much | `clip_grad_norm_` called on **scaled** gradients | call `scaler.unscale_(opt)` first |
| resumed run skips many steps | scaler state not restored | save/load `scaler.state_dict()` |
| `RuntimeError: unscale_() has already been called` | `unscale_` twice in one step | once per step |

In [11]:
# One last end-to-end proof of the first symptom check on this machine.
model, x, y = toy()
def dtypes_inside(dtype):
    with torch.autocast("cpu", dtype=dtype): p = model(x)
    return p.dtype
print("forward output dtype  fp32 loop:", model(x).dtype, "| bf16 autocast:", dtypes_inside(torch.bfloat16), "| fp16 autocast:", dtypes_inside(torch.float16))
check("the 10-second test: output dtype is fp32 / bf16 / fp16 for the three modes", (model(x).dtype, dtypes_inside(torch.bfloat16), dtypes_inside(torch.float16)) == (torch.float32, torch.bfloat16, torch.float16))

print(f"\n{'#':>2s}  {'status':6s}  check")
for i, (n, ok, d) in enumerate(CHECKS, 1): print(f"{i:2d}  {'PASS' if ok else 'FAIL':6s}  {n}")
failed = [n for n, ok, _ in CHECKS if not ok]
print(f"\n{len(CHECKS) - len(failed)} passed, {len(failed)} failed")
assert not failed, failed

forward output dtype  fp32 loop: torch.float32 | bf16 autocast: torch.bfloat16 | fp16 autocast: torch.float16
PASS - the 10-second test: output dtype is fp32 / bf16 / fp16 for the three modes 

 #  status  check
 1  PASS    baseline fp32 loop trains
 2  PASS    bf16 autocast loop trains
 3  PASS    outside the block autocast is off, inside it is on (bf16)
 4  PASS    the first Linear gets fp32 data in; every Linear outputs bf16
 5  PASS    later Linears already receive bf16 activations (the cast happens once, then stays cheap)
 6  PASS    weights, loss and gradients are fp32
 7  PASS    bf16 prediction differs from fp32 by ~0.1-3% (it really used bf16)
 8  PASS    fp16 autocast + GradScaler loop trains
 9  PASS    scaler.scale(loss) == loss * S
10  PASS    gradients are S times bigger before unscale
11  PASS    an inf gradient makes step() SKIP the update (weights unchanged)
12  PASS    ...and update() halves the scale (1024 -> 512)
13  PASS    without unscale_, clipping sees a norm S 

## Takeaways

1. **Turning on mixed precision = wrapping the forward pass in `torch.autocast`.** For bf16 that is the *only* change (plus `.float()` on the loss input).
2. **fp16 additionally needs a `GradScaler`:** `scale(loss).backward()` → `step(opt)` → `update()`; call `unscale_(opt)` before clipping.
3. **Weights, gradients, loss and optimizer stay fp32** - verified by printing dtypes. You never convert the model with `.half()`.
4. **One loop, three modes:** disabled autocast/scaler are no-ops, so a flag switches precision (Step 5). In this repo that flag is `train.amp.enabled` / `train.amp.dtype`.
5. **Verify, don't assume:** one batch and a dtype print tells you whether it is really on.

### Exercises
- Make `train()` accept `amp_dtype="auto"` and call `pick_precision` for you.
- In Step 3, set `init_scale=2**30` and watch how many steps are skipped and where `S` settles.
- Add gradient accumulation to the universal loop (divide the loss by the number of micro-batches; call `unscale_`/`step`/`update` once per optimizer step).
- Turn on `amp.enabled` in a copy of a HaleBlocks config and check that `amp/scale` and `amp/dtype` appear in the logged metrics.

### Honest limits
CPU only, a tiny model: this notebook shows **correctness and mechanics** (which lines, which dtypes, what the scaler does), **not speed** - the speed-up needs GPU tensor cores or similar hardware. The `cuda`/`mps` branch of `pick_precision`, the Hugging Face / Lightning / Accelerate snippets and the full `Trainer` were **not executed**.